In [66]:
# Import OS
import os

# Import the Pandas Library
import pandas as pd

# Import the matplotlib
import matplotlib.pyplot as plt

# Import numpy
import numpy as np

# Load in the CSV into a dataframe
df = pd.read_csv("../data/raw/Procurement_KPI_Analysis_Dataset.csv")

In [67]:
# Look at the first 5 rows to inspect the dataset
df.head()

,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No


In [68]:
# Find the shape of the data set
df.shape

(777, 11)

In [69]:
# Find out info for this dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PO_ID             777 non-null    str    
 1   Supplier          777 non-null    str    
 2   Order_Date        777 non-null    str    
 3   Delivery_Date     690 non-null    str    
 4   Item_Category     777 non-null    str    
 5   Order_Status      777 non-null    str    
 6   Quantity          777 non-null    int64  
 7   Unit_Price        777 non-null    float64
 8   Negotiated_Price  777 non-null    float64
 9   Defective_Units   641 non-null    float64
 10  Compliance        777 non-null    str    
dtypes: float64(3), int64(1), str(7)
memory usage: 66.9 KB


In [70]:
# Find exact columns names
df.columns.to_list()

['PO_ID',
 'Supplier',
 'Order_Date',
 'Delivery_Date',
 'Item_Category',
 'Order_Status',
 'Quantity',
 'Unit_Price',
 'Negotiated_Price',
 'Defective_Units',
 'Compliance']

In [71]:
# Standardize columns names removing trailing and leading white space
df.columns = df.columns.str.strip()

# Convert all columns to lower case for SQL 
df.columns = df.columns.str.lower()

# Replace any comma with underscore
df.columns = df.columns.str.replace(",","_")

In [72]:
# Verify the above code to ensure it worked
df.columns.tolist()

['po_id',
 'supplier',
 'order_date',
 'delivery_date',
 'item_category',
 'order_status',
 'quantity',
 'unit_price',
 'negotiated_price',
 'defective_units',
 'compliance']

In [73]:
# Update the order date and delivery date to convert to date time
df["order_date"] = pd.to_datetime(df["order_date"])
df["delivery_date"] = pd.to_datetime(df["delivery_date"])

# Check that the conversion worked
df[["order_date", "delivery_date"]].dtypes

order_date       datetime64[us]
delivery_date    datetime64[us]
dtype: object

In [74]:
# Find the missing values in each column
df.isna().sum()

po_id                 0
supplier              0
order_date            0
delivery_date        87
item_category         0
order_status          0
quantity              0
unit_price            0
negotiated_price      0
defective_units     136
compliance            0
dtype: int64

In [75]:
# Find summary statistics for the numerical columns rounding 2 deciaml places
df.describe().round(2)

,order_date,delivery_date,quantity,unit_price,negotiated_price,defective_units
count,777,690,777.00,777.00,777.00,641.00
mean,2022-12-28 23:15:31.274131,2023-01-12 03:16:10.434782,1094.66,58.28,53.66,74.80
min,2022-01-01 00:00:00,2022-01-06 00:00:00,51.00,10.84,9.27,0.00
25%,2022-07-02 00:00:00,2022-07-19 18:00:00,615.00,33.29,30.46,26.00
50%,2022-12-24 00:00:00,2023-01-19 12:00:00,1075.00,58.95,53.80,49.00
75%,2023-07-07 00:00:00,2023-07-18 00:00:00,1548.00,83.13,76.55,100.00
max,2024-01-01 00:00:00,2024-01-12 00:00:00,5000.00,109.17,107.39,321.00
std,NaN,NaN,647.84,28.10,26.09,69.19


In [76]:
# Look for unique values within the column
df.nunique()

po_id               777
supplier              5
order_date          476
delivery_date       449
item_category         5
order_status          4
quantity            623
unit_price          747
negotiated_price    752
defective_units     202
compliance            2
dtype: int64

In [77]:
# Look for the value counts for text columns
df["order_status"].value_counts()

order_status
Delivered              560
Pending                 81
Partially Delivered     73
Cancelled               63
Name: count, dtype: int64

In [78]:
df["supplier"].value_counts()

supplier
Delta_Logistics    171
Epsilon_Group      166
Beta_Supplies      156
Gamma_Co           143
Alpha_Inc          141
Name: count, dtype: int64

In [79]:
df["item_category"].value_counts()

item_category
Office Supplies    174
MRO                164
Electronics        152
Packaging          148
Raw Materials      139
Name: count, dtype: int64

In [80]:
df["compliance"].value_counts()

compliance
Yes    640
No     137
Name: count, dtype: int64

In [81]:
# Look at order status where the delivery date is missing
df[df["delivery_date"].isna()]["order_status"].value_counts()

order_status
Delivered              68
Cancelled               8
Pending                 6
Partially Delivered     5
Name: count, dtype: int64

In [82]:
# Look at order status where the defective units are missing
df[df["defective_units"].isna()]["order_status"].value_counts()

order_status
Delivered              102
Partially Delivered     13
Pending                 12
Cancelled                9
Name: count, dtype: int64

**Now that we can see the missing delivery dates and the defective units missing, we need to see what suppliers are impacted the most by the missing data**

In [83]:
# Count the missing data for the delivery dates per supplier
df[df["delivery_date"].isna()]["supplier"].value_counts()

supplier
Alpha_Inc          24
Delta_Logistics    20
Epsilon_Group      17
Beta_Supplies      13
Gamma_Co           13
Name: count, dtype: int64

In [84]:
# Count the missing data for the defective units per suppler
df[df["defective_units"].isna()]["supplier"].value_counts()

supplier
Delta_Logistics    44
Alpha_Inc          28
Epsilon_Group      25
Beta_Supplies      24
Gamma_Co           15
Name: count, dtype: int64

**Now that we have a better understanding on where the missing data comes from, next step needs to be whether these orders were actually delivered**

In [85]:
# Look for the delivered orders that are missing delivery date
df[
    (df["order_status"] == 'Delivered') &
    (df["delivery_date"].isna())
]

,po_id,supplier,order_date,delivery_date,item_category,order_status,quantity,unit_price,negotiated_price,defective_units,compliance
13,PO-00014,Beta_Supplies,2022-02-02,NaT,MRO,Delivered,5000,18.30,16.88,NaN,Yes
29,PO-00030,Epsilon_Group,2023-08-27,NaT,MRO,Delivered,1005,80.55,73.31,30.0,Yes
39,PO-00040,Epsilon_Group,2022-12-11,NaT,Office Supplies,Delivered,524,46.28,39.65,14.0,Yes
45,PO-00046,Alpha_Inc,2022-12-19,NaT,Electronics,Delivered,1025,21.11,19.25,15.0,Yes
51,PO-00052,Delta_Logistics,2022-05-08,NaT,Office Supplies,Delivered,612,20.99,19.75,102.0,Yes
...,...,...,...,...,...,...,...,...,...,...,...
747,PO-00748,Epsilon_Group,2022-06-06,NaT,Office Supplies,Delivered,1737,24.88,24.28,46.0,Yes
750,PO-00751,Delta_Logistics,2023-04-30,NaT,Electronics,Delivered,1809,13.58,12.68,NaN,No
752,PO-00753,Epsilon_Group,2022-04-28,NaT,MRO,Delivered,1853,21.92,20.72,NaN,Yes
769,PO-00770,Epsilon_Group,2023-08-13,NaT,Office Supplies,Delivered,748,85.95,79.16,32.0,Yes


In [86]:
# Look for the delivered orders that are missing defective units
df[
    (df["order_status"] == "Delivered") &
    (df["defective_units"].isna())
]

,po_id,supplier,order_date,delivery_date,item_category,order_status,quantity,unit_price,negotiated_price,defective_units,compliance
13,PO-00014,Beta_Supplies,2022-02-02,NaT,MRO,Delivered,5000,18.30,16.88,NaN,Yes
15,PO-00016,Beta_Supplies,2022-04-06,2022-04-21,Packaging,Delivered,5000,15.41,15.12,NaN,Yes
17,PO-00018,Epsilon_Group,2022-08-27,2022-09-04,Raw Materials,Delivered,921,51.48,50.61,NaN,Yes
19,PO-00020,Delta_Logistics,2023-09-09,2023-09-24,Raw Materials,Delivered,180,45.74,38.89,NaN,No
21,PO-00022,Delta_Logistics,2023-07-29,2023-08-11,Raw Materials,Delivered,1382,24.93,23.75,NaN,Yes
...,...,...,...,...,...,...,...,...,...,...,...
730,PO-00731,Alpha_Inc,2022-09-16,2022-09-18,Office Supplies,Delivered,778,65.85,62.08,NaN,No
739,PO-00740,Delta_Logistics,2022-03-12,2022-03-19,MRO,Delivered,241,87.42,82.15,NaN,Yes
750,PO-00751,Delta_Logistics,2023-04-30,NaT,Electronics,Delivered,1809,13.58,12.68,NaN,No
752,PO-00753,Epsilon_Group,2022-04-28,NaT,MRO,Delivered,1853,21.92,20.72,NaN,Yes


In [87]:
# Create a sperate dataframe that holds data that does not have missing units
quality_df = df[
    (df["order_status"] == "Delivered") &
    (df["delivery_date"].notna()) &
    (df["defective_units"].notna())
].copy()

# Display the first 5 rows of the new dataset
quality_df.head()

,po_id,supplier,order_date,delivery_date,item_category,order_status,quantity,unit_price,negotiated_price,defective_units,compliance
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No
5,PO-00006,Epsilon_Group,2022-08-17,2022-08-29,MRO,Delivered,1145,69.21,63.57,39.0,Yes


**Now that we have looked into the missing data, we need to understand what compliance is without guessing what it means. We cannot make an assumption about the definittion, but we can look at if it has any relationship to delivery and defective units.**

In [88]:
# Run the crosstab function to see the compliance status with order status
pd.crosstab(
    df["order_status"],
    df["compliance"]
)

compliance,No,Yes
order_status,,
Cancelled,11,52
Delivered,96,464
Partially Delivered,17,56
Pending,13,68


In [89]:
# Look at statistics on the defective units
df.groupby("compliance")["defective_units"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
compliance,,,
No,102,111.84,90.5
Yes,539,67.79,44.0


In [108]:
# Find the difference in the actual and expected delivery date
df["delivery_lead_time"] = (
    df["delivery_date"] - df["order_date"]
).dt.days

# Display the first few values
df[["order_date", "delivery_date", "delivery_lead_time"]].head()

,order_date,delivery_date,delivery_lead_time
0,2023-10-17,2023-10-25,8.0
1,2022-04-25,2022-05-05,10.0
2,2022-01-26,2022-02-15,20.0
3,2022-10-09,2022-10-28,19.0
4,2022-09-08,2022-09-20,12.0


In [91]:
# Compare average delivery lead time between compliant and non-compliant orders
df.groupby("compliance")["delivery_lead_time"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
compliance,,,
No,125,10.36,11.0
Yes,565,10.87,11.0


**Based on the calculations above, there seems to be a relationship between the defective units and whether the order qualifies as compliant. In similar calculations there seems to be no relationship between compliance between order date and delivery date. This data did not provide expected date, so late deliveries cannot be calculated. Now we need to calculate the defective rate as these orders may have had a significant number of units that might skew the data**

In [92]:
# Calculate the defective unit rate\
quality_df["defect_rate"] = (
    (quality_df["defective_units"] / quality_df["quantity"])
) * 100

In [93]:
# Compare these defective rates between compliant and non-compliant orders
quality_df.groupby("compliance")["defect_rate"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
compliance,,,
No,63,10.09,10.15
Yes,339,6.20,4.55


**The above calculations show the relationship between the non-compliance orders and defects of the units. As shown, the average defective rate for those orders that were marked non-compliant was 10.09% versus those orders that were compliant at 6.2%. Now the next step is to see which suppliers have provided the most cost savings.**

In [94]:
# Verify there are no list price higher than the net price
df[df["negotiated_price"] > df["unit_price"]]

,po_id,supplier,order_date,delivery_date,item_category,order_status,quantity,unit_price,negotiated_price,defective_units,compliance,delivery_lead_time


In [95]:
# Calculate the savings per unit
df["savings_per_unit"] = (
    df["unit_price"] - df["negotiated_price"]
).round(2)

# Calculate the total savings achieved per order
df["total_savings"] = (
    df["savings_per_unit"] * df["quantity"]
).round(2)

# Calculate the percentage savings comapred to list price
df["savings_percentage"] = (
    df["savings_per_unit"] / df["unit_price"]
).round(4)*100


In [96]:
# Display these findings
df[
    [
    "supplier",
    "quantity",
    "unit_price",
    "negotiated_price",
    "savings_per_unit",
    "total_savings",
    "savings_percentage"
    ]
].head()

,supplier,quantity,unit_price,negotiated_price,savings_per_unit,total_savings,savings_percentage
0,Alpha_Inc,1176,20.13,17.81,2.32,2728.32,11.53
1,Delta_Logistics,1509,39.32,37.34,1.98,2987.82,5.04
2,Gamma_Co,910,95.51,92.26,3.25,2957.50,3.40
3,Beta_Supplies,1344,99.85,95.52,4.33,5819.52,4.34
4,Delta_Logistics,1180,64.07,60.53,3.54,4177.20,5.53


In [97]:
# Now check cost savings by supplier
supplier_savings = (
    df.groupby("supplier").agg(
        total_savings=("total_savings", "sum"),
        average_savings_percentage=("savings_percentage", "mean")
        
    ).sort_values("total_savings", ascending=False)
)

# Display this variable rounding to 2 decimal places
supplier_savings.round(2)

,total_savings,average_savings_percentage
supplier,,
Beta_Supplies,889940.89,7.83
Epsilon_Group,844980.18,8.04
Delta_Logistics,781976.49,7.81
Gamma_Co,725308.42,7.98
Alpha_Inc,688920.49,8.21


**Now that we have an idea on cost savings for these suppliers, we need to see what the weighted calculations are as these numbers above do not factor in orders per supplier.**

In [98]:
# Calculate total list price value for each order
df["total_list_value"] = (
    df["unit_price"] * df["quantity"]
)

df["actual_spend"] = (
    df["negotiated_price"] * df["quantity"]
)

# Calculate total list value and total savings for each supplier
supplier_savings = (
    df.groupby("supplier").agg(
        total_list_value=("total_list_value", "sum"),
        actual_spend=("actual_spend", "sum"),
        total_savings=("total_savings", "sum")
    )
)

# Now calculate each suppliers overall savings rate
supplier_savings["savings_rate"] = (
    supplier_savings["total_savings"] / supplier_savings["total_list_value"]
)*100

# Sort suppliers by total savings
supplier_savings = supplier_savings.sort_values(
    "total_savings", ascending=False
)

# Display the supplier savings result
supplier_savings.round(2)

,total_list_value,actual_spend,total_savings,savings_rate
supplier,,,,
Beta_Supplies,10748606.79,9858665.90,889940.89,8.28
Epsilon_Group,10696136.24,9851156.06,844980.18,7.90
Delta_Logistics,10018216.96,9236240.47,781976.49,7.81
Gamma_Co,9313230.13,8587921.71,725308.42,7.79
Alpha_Inc,8528632.74,7839712.25,688920.49,8.08


**Now that we have basic supplier metrics, it is time to start creating a table that is used as the supplier scorecard. This data will then be used in PowerBI to display metrics and make comparisons**

In [99]:
# Calculate the supplier performance metrics
supplier_performance = (
    df.groupby("supplier").agg(
        total_orders=("supplier", "count"),
        actual_spend = ("actual_spend", "sum"),
        total_savings = ("total_savings", "sum"),
        savings_rate = ("savings_percentage", "mean"),
        average_lead_time = ("delivery_lead_time", "mean")
    )
)

# Display this summary'
supplier_performance.round(2)

,total_orders,actual_spend,total_savings,savings_rate,average_lead_time
supplier,,,,,
Alpha_Inc,141,7839712.25,688920.49,8.21,10.61
Beta_Supplies,156,9858665.90,889940.89,7.83,11.27
Delta_Logistics,171,9236240.47,781976.49,7.81,10.85
Epsilon_Group,166,9851156.06,844980.18,8.04,10.87
Gamma_Co,143,8587921.71,725308.42,7.98,10.19


In [100]:
# Calculate the average defect rate for each supplier
supplier_quality = (
    quality_df.groupby("supplier")["defect_rate"]
    .mean()
    .rename("average_defect_rate")
)

# Add the quality metric to the supplier performance table
supplier_performance = supplier_performance.join(
    supplier_quality
)

In [101]:
# Calculate the percentage of compliant orders for each supplier
supplier_compliance = (
    df.groupby("supplier")["compliance"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .rename("compliance_rate")
)

# Add the compliance rate to the supplier performance table
supplier_performance = supplier_performance.join(
    supplier_compliance
)

# Display the completed supplier summary
supplier_performance.round(2)

,total_orders,actual_spend,total_savings,savings_rate,average_lead_time,average_defect_rate,compliance_rate
supplier,,,,,,,
Alpha_Inc,141,7839712.25,688920.49,8.21,10.61,2.09,93.62
Beta_Supplies,156,9858665.90,889940.89,7.83,11.27,9.89,75.64
Delta_Logistics,171,9236240.47,781976.49,7.81,10.85,14.55,60.82
Epsilon_Group,166,9851156.06,844980.18,8.04,10.87,3.11,98.19
Gamma_Co,143,8587921.71,725308.42,7.98,10.19,5.04,86.01


**Now that we have all the data we need, it is time to create our final table and export**

In [102]:
# Create the final dataset for Power BI or Tableau
analysis_df = df[
    [
        "supplier",
        "item_category",
        "order_status",
        "compliance",
        "quantity",
        "order_date",
        "delivery_date",
        "delivery_lead_time",
        "unit_price",
        "negotiated_price",
        "actual_spend",
        "savings_per_unit",
        "total_savings",
        "savings_percentage",
        "defective_units"
    ]
].copy()

In [103]:
# Calculate defect rate only for delivered orders with valid quality data
analysis_df["defect_rate"] = np.where(
    (analysis_df["order_status"] == "Delivered") &
    (analysis_df["delivery_date"].notna()) &
    (analysis_df["defective_units"].notna()),
    (analysis_df["defective_units"] / analysis_df["quantity"]) * 100,
    np.nan
)

In [104]:
# Export the order-level cleaned dataset
analysis_df.to_csv(
    "../data/processed/supplier_performance_cleaned.csv",
    index=False
)

In [105]:
# Export the supplier-level performance summary
supplier_performance.to_csv(
    "../data/processed/supplier_performance_summary.csv",index=False
)

In [106]:
# Confirm the shape of the final order-level dataset
print("Order-level dataset:", analysis_df.shape)

# Confirm the shape of the supplier summary dataset
print("Supplier summary dataset:", supplier_performance.shape)

Order-level dataset: (777, 16)
Supplier summary dataset: (5, 7)


In [107]:
# Add the SQL database functions to use SQL
import sqlite3
conn = sqlite3.connect("../data/processed/supplier_performance.db")
analysis_df.to_sql("supplier_performance", conn, if_exists="replace", index=False)
conn.close()